In [1]:
import pandas as pd

# 讀取四個描述檔案
descriptions = {}
artwork_ids = ["caravaggio_calling", "rublev_trinity", "raphael_athens", "bhmpi_anonymous"]

for artwork_id in artwork_ids:
    with open(f"data/descriptions/{artwork_id}.txt", "r", encoding="utf-8") as f:
        descriptions[artwork_id] = f.read()

print("描述檔案載入完成：")
for k, v in descriptions.items():
    print(f"  {k}: {len(v)} characters")

描述檔案載入完成：
  caravaggio_calling: 2070 characters
  rublev_trinity: 1914 characters
  raphael_athens: 2063 characters
  bhmpi_anonymous: 1976 characters


In [2]:
import re

def split_into_claims(text):
    # 以句號、問號、驚嘆號切分
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    claims = []
    for s in sentences:
        s = s.strip()
        if len(s) > 20:  # 過濾太短的片段
            claims.append(s)
    return claims

# 切分所有描述
all_claims = {}
for artwork_id in artwork_ids:
    claims = split_into_claims(descriptions[artwork_id])
    all_claims[artwork_id] = claims
    print(f"{artwork_id}: {len(claims)} claim units")

caravaggio_calling: 13 claim units
rublev_trinity: 12 claim units
raphael_athens: 16 claim units
bhmpi_anonymous: 16 claim units


In [3]:
# 建立所有 claim units 的 DataFrame
rows = []
claim_counter = 1

for artwork_id in artwork_ids:
    for i, claim in enumerate(all_claims[artwork_id]):
        rows.append({
            "claim_id": f"C{claim_counter:03d}",
            "artwork_id": artwork_id,
            "claim_index": i + 1,
            "claim_text": claim,
            "claim_type": "",        # VISUAL / HISTORICAL / INTERPRETIVE / ATTRIBUTIVE
            "evidence_source": "",   # IMAGE_VISIBLE / METADATA / SCHOLARLY / UNTRACED
            "accuracy": "",          # GROUNDED / PLAUSIBLE / HALLUCINATED / OMISSION
            "hedging": ""            # HEDGED / UNHEDGED
        })
        claim_counter += 1

df = pd.DataFrame(rows)
print(f"總共 {len(df)} 個 claim units")
print("\n前三筆預覽：")
df[["claim_id", "artwork_id", "claim_text"]].head(3)

總共 57 個 claim units

前三筆預覽：


,claim_id,artwork_id,claim_text
0,C001,caravaggio_calling,This painting depicts a dimly lit interior sce...
1,C002,caravaggio_calling,"From the right side of the composition, two st..."
2,C003,caravaggio_calling,A sharp diagonal beam of light enters from the...


In [4]:
# 存成 CSV
df.to_csv("data/annotation_template.csv", index=False, encoding="utf-8-sig")
print("已儲存：data/annotation_template.csv")
print("\n接下來：")
print("1. 用 Excel 打開這個 CSV 檔案")
print("2. 對每一個 claim unit 填入四個欄位：")
print("   claim_type:    VISUAL / HISTORICAL / INTERPRETIVE / ATTRIBUTIVE")
print("   evidence_source: IMAGE_VISIBLE / METADATA / SCHOLARLY / UNTRACED")
print("   accuracy:      GROUNDED / PLAUSIBLE / HALLUCINATED / OMISSION")
print("   hedging:       HEDGED / UNHEDGED")
print("3. 存檔後回來告訴我")

已儲存：data/annotation_template.csv

接下來：
1. 用 Excel 打開這個 CSV 檔案
2. 對每一個 claim unit 填入四個欄位：
   claim_type:    VISUAL / HISTORICAL / INTERPRETIVE / ATTRIBUTIVE
   evidence_source: IMAGE_VISIBLE / METADATA / SCHOLARLY / UNTRACED
   accuracy:      GROUNDED / PLAUSIBLE / HALLUCINATED / OMISSION
   hedging:       HEDGED / UNHEDGED
3. 存檔後回來告訴我


In [5]:
# 讀入標注完成的 CSV
df_annotated = pd.read_csv("data/annotation_template.csv", encoding="utf-8-sig")

# 確認標注完整性
total = len(df_annotated)
filled = df_annotated["accuracy"].notna().sum()
print(f"總共 {total} 個 claim units，已標注 {filled} 個")
print()
print("各作品標注數量：")
print(df_annotated.groupby("artwork_id")["claim_id"].count())

總共 57 個 claim units，已標注 56 個

各作品標注數量：
artwork_id
bhmpi_anonymous       16
caravaggio_calling    13
raphael_athens        16
rublev_trinity        12
Name: claim_id, dtype: int64
